Burada data_collectionda belirlenen aday soruların her biri için 10 adet sorular sorulmakta ve elde edilen veriler kayıt altına alınmaktadır.

In [ ]:
# Modeli ve performansı artırmak için gerekli kütüphaneleri yüklenmekte.
!pip install -q transformers accelerate

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, LogitsProcessor
from google.colab import drive
import torch, torch.nn.functional as F, re, json, time, os, numpy as np
import random

# Drive üzerinde yer alan veriye erişim sağlanmakta, ardından verilerin kayıt edileceği klasör oluşturulmakta.
drive.mount('/content/drive')
os.makedirs('/content/drive/MyDrive/proje5/data', exist_ok=True)

In [ ]:
# Modelimiz
model_name = "Qwen/Qwen3.5-4B"
tokenizer  = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.bfloat16, device_map="auto")
model.eval()

In [ ]:
# data collection kısmında yaptığımız metodların aynısı burada da kullanıyoruz. 
# Tekrar yazılmasının nedeni her notebook kendi başına çalışabilmesi için.
# Latex formatında sayı dönüşümlerinde meydana gelen nokta virgül dönüşümü yapılması
def noktalı_sayi_isle(s):
    s = s.replace(',', '.')
    n = s.count('.')
    if n == 0:
        try: return float(s)
        except: return None
    elif n == 1:
        sol, sag = s.split('.')
        if sol == '0':
            try: return float(s)
            except: return None
        elif len(sag) == 3:
            try: return float(sol+sag)
            except: return None
        else:
            try: return float(s)
            except: return None
    elif n == 2:
        try: return float(s.replace('.',''))
        except: return None
    return None

# Bazı cevaplarda sayısal sonuçlar yerine gelen ifadelere karşılık gelen sonuçların None olarak işaretlenmesi
def son_sayi_cek(metin):
    son = metin.strip().split('\n')[-1]
    for k in ['belirlenemez','bilgi yok','bilgi verilmedi','yetersiz bilgi',
              'daha fazla bilgi','hesaplanamaz','mümkün değil','tanımsız',
              'bilinmiyor','tutarsız','kontrol edin','gözden geçir']:
        if k in son.lower(): return None
    sayilar = re.findall(r'-?\d+(?:[.,]\d+)*', son)
    return noktalı_sayi_isle(sayilar[-1]) if sayilar else None

# GT ile üretilen cevabın kontrol edilmesi.
def is_correct(uretilen, gercek):
    p, t = son_sayi_cek(uretilen), son_sayi_cek(gercek)
    if p is None or t is None: return False
    return abs(p - t)

In [ ]:
# Ham veri toplama bu class Ai ile hazırlanmıştır.
class HamLogitYakalayici(LogitsProcessor):
    def __init__(self):
        self.ham_logitler = []
    # şimdiye kadar üretilen her token için ham logitleri kaydeder ve geri döndürür
    def __call__(self, input_ids, scores):
        self.ham_logitler.append(scores[0].detach().cpu())
        return scores

# moving average hesaplayan metod burada kullanılan metriklerin daha stabil gözlemlenmesi için hareketli ortalama hesaplanmakta.
# window size 5 olarak belirlendi, bu isteğe göre değiştirilebilir. top10 suma da uygun olması açısından 5 belirlendi.
def _moving_avg(lst, w=5):
    arr = np.array(lst)
    return [round(float(arr[max(0, i-w+1):i+1].mean()), 6) for i in range(len(arr))]

# Token sayısını 200 ile sınırladım çünkü denem çalışmalarında gördüğüm üzere 200'den sonra model yanlış cevap üretmeye başlıyor.
# Temperature'ı 1.0 olarak bıraktım çeşitlilik artırmak ve düzgün cevaplar almak için trade-off olarak.
def uret_detayli(soru, max_new_tokens=200, temperature=1.0):
    # Modelin beklediği formata uygun şekilde prompt hazırlandım modelin beklediği formatı Ai ile öğrenip hazırladım.
    # Derin düşünmeyi kapattım bazı durumlarda çok uzun zamanlar alıyordu cevaplar için. Tokenize etme, string olarak döndürür (tokenize = False).
    messages = [{"role": "user", "content": soru}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)

    # Burada prompt tensörlere dönüştürülüyor.
    inputs    = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[1]

    # ham logitler atanmakta.
    # LogitsProcessor kullanılarak top_k/top_p filtresinden ÖNCE ham logitler yakalanmaktadır.
    # output_scores=True ile gelen logitler filtrelenmiş olduğundan bu yöntem tercih edilmiştir.
    yakalayici = HamLogitYakalayici()
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            temperature=temperature, top_p=0.95, top_k=20,
            min_p=0.0, do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            logits_processor=[yakalayici]
        )
    # prompt kısmını keserek sadece üretilen kısmı alınmakta.
    uretilen_ids = outputs[0][input_len:]
    # tokenler string dönüştürülüyor ileride cevabı daha rahat gözlemlemek için.
    cevap = tokenizer.decode(uretilen_ids, skip_special_tokens=True)

    # literatürde kullanılan metrikler bu metrikleri Ai ile belirledim ve ham logitlerden bu metrikleri hesaplıyorum.
    top10_sums, top1_probs, top1_top2_fark, entropy_list = [], [], [], []
    for ham_logit in yakalayici.ham_logitler:
        probs = F.softmax(ham_logit, dim=-1)
        topk  = torch.topk(probs, k=10)
        top10_sums.append(round(topk.values.sum().item(), 6))
        top1_probs.append(round(topk.values[0].item(), 6))
        top1_top2_fark.append(round((topk.values[0]-topk.values[1]).item(), 6))
        p_log_p = probs * torch.log(probs + 1e-10)
        entropy_list.append(round(-p_log_p.sum().item(), 4))
    return {
        "cevap": cevap, "top10_sums": top10_sums,
        "top1_probs": top1_probs, "top1_top2_fark": top1_top2_fark,
        "entropy": entropy_list, "num_tokens": len(top10_sums),
        "ma_top10_sums"    : _moving_avg(top10_sums),
        "ma_top1_probs"    : _moving_avg(top1_probs),
        "ma_top1_top2_fark": _moving_avg(top1_top2_fark),
        "ma_entropy"       : _moving_avg(entropy_list),
    }

print("Fonksiyonlar tanımlandı")

In [ ]:
# Sorular tarama_adaylar.json'dan okunmaktadır.
# Dataset'e ihtiyaç yoktur — soru metinleri ve ground truth cevaplar bu dosyada mevcuttur.
TARAMA_KAYIT = '/content/drive/MyDrive/proje5/data/tarama_adaylar.json'
DETAY_KAYIT  = '/content/drive/MyDrive/proje5/data/veri_500_ma.json'

with open(TARAMA_KAYIT, encoding='utf-8') as f:
    tarama = json.load(f)

adaylar = tarama['adaylar']

# Veri setinde soruların zorluğu sıralama ile ilişkili olabileceğinden karıştırma işlemi yapıyorum
# 500 aday al, rastgele karıştır ve 250 eğitim + 250 test böl
random.seed(37)
random.shuffle(adaylar)
adaylar = adaylar[:500]

# ilk 250 eğitim son 250 test setidir.
egitim = adaylar[:250]
test   = adaylar[250:]

print(f"Toplam aday : {len(adaylar)}")
print(f"Eğitim      : {len(egitim)}")
print(f"Test        : {len(test)}")

In [ ]:
# Ai ile eğitim ve test setlerinin durumunu kontrol eden ve kaydeden bir mekanizma hazırladım böylece kaldığı yerden devam edebiliriz.
if os.path.exists(DETAY_KAYIT):
    with open(DETAY_KAYIT, encoding='utf-8') as f:
        kayit = json.load(f)
    tamamlanan_egitim = kayit['egitim']
    tamamlanan_test   = kayit['test']
    print(f"⚡ Devam: Eğitim {len(tamamlanan_egitim)}/250, "
          f"Test {len(tamamlanan_test)}/250")
else:
    tamamlanan_egitim = []
    tamamlanan_test   = []
    print("🆕 Sıfırdan başlıyor")

In [ ]:
# Her soru için 10 farklı cevap üreten ve sonuçları kayıt altına alma işlemi burada yapılmakta.
def isle(soru_listesi, tamamlananlar, split_adi):
    t0        = time.time()
    baslangic = len(tamamlananlar)

    for i in range(baslangic, len(soru_listesi)):
        item   = soru_listesi[i]
        soru   = item['soru']
        gercek = item['gercek_cevap']

        cevaplar = []
        # Her soru için 10 cevap ve bu cevapları karşılaştırılması.
        for _ in range(10):
            sonuc             = uret_detayli(soru)
            sonuc['dogru_mu'] = is_correct(sonuc['cevap'], gercek)
            cevaplar.append(sonuc)

        dogru_sayisi = sum(c['dogru_mu'] for c in cevaplar)

        tamamlananlar.append({
            "idx"         : item['idx'],
            "split"       : split_adi,
            "soru"        : soru,
            "gercek_cevap": gercek,
            "dogru_sayisi": dogru_sayisi,
            "cevaplar"    : cevaplar
        })

        # Bağlantı kopmasına karşı her soru sonunda Drive'a kaydedilmekte.
        with open(DETAY_KAYIT, 'w', encoding='utf-8') as f:
            json.dump({"egitim": tamamlanan_egitim, "test": tamamlanan_test},
                      f, ensure_ascii=False, indent=2)

        gecen = time.time() - t0
        kalan = (gecen / (i - baslangic + 1)) * (len(soru_listesi) - i - 1) if i > baslangic else 0
        print(f"[{split_adi}] [{i+1:3d}/{len(soru_listesi)}] "
              f"idx={item['idx']:4d} | Doğru: {dogru_sayisi}/10 | "
              f"Geçen: {gecen/60:.1f}dk | ~Kalan: {kalan/60:.0f}dk")

    return tamamlananlar

In [ ]:
print("EĞİTİM SETİ (250 soru * 10 cevap = 2500 inference)")
tamamlanan_egitim = isle(egitim, tamamlanan_egitim, "egitim")

print("\nTEST SETİ (250 soru * 10 cevap = 2500 inference)")
tamamlanan_test = isle(test, tamamlanan_test, "test")

# Üretilen verinin özet istatistikleri — veri kalitesini kontrol etmek için
with open(DETAY_KAYIT, encoding='utf-8') as f:
    veri = json.load(f)

for split in ['egitim', 'test']:
    sorular       = veri[split]
    toplam_cevap  = sum(len(s['cevaplar']) for s in sorular)
    toplam_dogru  = sum(c['dogru_mu'] for s in sorular for c in s['cevaplar'])
    toplam_yanlis = toplam_cevap - toplam_dogru
    ort_token     = np.mean([c['num_tokens'] for s in sorular for c in s['cevaplar']])

    print(f"{split.upper()} SETİ:")
    print(f"  Soru sayısı  : {len(sorular)}")
    print(f"  Toplam cevap : {toplam_cevap}")
    print(f"  Doğru        : {toplam_dogru} (%{toplam_dogru/toplam_cevap*100:.1f})")
    print(f"  Yanlış       : {toplam_yanlis} (%{toplam_yanlis/toplam_cevap*100:.1f})")
    print(f"  Ort. token   : {ort_token:.0f}")
    print()